# Lib1 scratch sweep comparison (basic vs weighted)

This notebook inspects local cached outputs for two sweeps (basic and weighted loss), diagnoses failed runs from logs, and evaluates each available best checkpoint on validation/test sets for direct comparison.

In [1]:
import os
import re
import json
import math
import inspect
from pathlib import Path

import yaml
import torch
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in os.sys.path:
    os.sys.path.insert(0, PROJECT_ROOT)

import boda
from boda.graph.utils import r2_score as graph_r2_score

In [2]:
LEARN_DIR = Path(PROJECT_ROOT) / "src" / "learn"
WANDB_ROOT = LEARN_DIR / "wandb"

SWEEP_CONFIGS = {
    "basic": {
        "sweep_id": "59fagojd",
        "logger_project": "bashor_lib1_scratch_basic",
        "metric": "epoch_end_val_r2",
    },
    "weighted": {
        "sweep_id": "tsdqtfnj",
        "logger_project": "bashor_lib1_scratch_weighted",
        "metric": "val_r2_score",
    },
}

print(f"LEARN_DIR: {LEARN_DIR}")
print(f"WANDB_ROOT: {WANDB_ROOT}")
print("Sweep setup:")
for mode, cfg in SWEEP_CONFIGS.items():
    print(f"  {mode:8s} sweep={cfg['sweep_id']} logger_project={cfg['logger_project']} metric={cfg['metric']}")

LEARN_DIR: /home/minhang/synBio_AL/boda2_EU/src/learn
WANDB_ROOT: /home/minhang/synBio_AL/boda2_EU/src/learn/wandb
Sweep setup:
  basic    sweep=59fagojd logger_project=bashor_lib1_scratch_basic metric=epoch_end_val_r2
  weighted sweep=tsdqtfnj logger_project=bashor_lib1_scratch_weighted metric=val_r2_score


In [3]:
def _run_id_from_dir(run_dir: Path) -> str:
    return run_dir.name.rsplit("-", 1)[-1]


def _read_text(path: Path) -> str:
    if not path.exists():
        return ""
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        return ""


def _read_yaml(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle)
    return data or {}


def _read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    return data or {}


def _cfg_value(config: dict, key: str):
    value = config.get(key)
    if isinstance(value, dict) and "value" in value:
        return value["value"]
    return value


def _extract_best_ckpt_path(output_log: str):
    matches = re.findall(r"Best model stashed at: (.+)", output_log)
    if not matches:
        return None
    best_rel = matches[-1].strip()
    if best_rel.startswith("./"):
        return str((LEARN_DIR / best_rel[2:]).resolve())
    return best_rel


def _extract_wandb_run_id(output_log: str):
    patterns = [
        r"Generated run_id: ([a-z0-9]+)",
        r"Initialized Wandb logging with run ID: ([a-z0-9]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, output_log)
        if matches:
            return matches[-1]
    return None


def _extract_sweep_ids(debug_log: str) -> list:
    ids = set(re.findall(r"/sweeps/([a-z0-9]+)", debug_log))
    ids.update(re.findall(r"wandb agent [^\n\s]*/[^\n\s]*/([a-z0-9]+)", debug_log))
    ids.update(re.findall(r"['\"]sweep_id['\"]:\s*['\"]([a-z0-9]+)['\"]", debug_log))
    ids.update(re.findall(r"sweep-([a-z0-9]+)/config-[a-z0-9]+\.yaml", debug_log))
    return sorted(ids)


def _build_run_to_sweeps_from_sweep_dir(wandb_root: Path) -> dict:
    run_to_sweeps = {}
    for path in wandb_root.glob("sweep-*/config-*.yaml"):
        sweep_id = path.parent.name.replace("sweep-", "")
        run_id = path.stem.replace("config-", "")
        run_to_sweeps.setdefault(run_id, set()).add(sweep_id)
    return {rid: sorted(sids) for rid, sids in run_to_sweeps.items()}


def _extract_failure_type(output_log: str) -> str:
    if "No space left on device" in output_log:
        return "no_space_left"
    if "unexpected end of data" in output_log:
        return "tar_unexpected_end_of_data"
    if "Traceback" in output_log:
        return "other_traceback"
    return "none"


def _run_record(run_dir: Path) -> dict:
    run_id = _run_id_from_dir(run_dir)
    output_log = _read_text(run_dir / "files" / "output.log")
    debug_log = _read_text(run_dir / "logs" / "debug.log")
    config = _read_yaml(run_dir / "files" / "config.yaml")
    summary = _read_json(run_dir / "files" / "wandb-summary.json")

    best_ckpt_path = _extract_best_ckpt_path(output_log)
    best_ckpt_exists = bool(best_ckpt_path and Path(best_ckpt_path).is_file())

    return {
        "run_dir": str(run_dir),
        "run_id": run_id,
        "wandb_run_id": _extract_wandb_run_id(output_log),
        "logger_project": _cfg_value(config, "logger_project"),
        "model_module": _cfg_value(config, "model_module"),
        "graph_module": _cfg_value(config, "graph_module"),
        "train_size_frac": _cfg_value(config, "train_size_frac"),
        "train_min_barcodes": _cfg_value(config, "train_min_barcodes"),
        "use_reverse_complements": _cfg_value(config, "use_reverse_complements"),
        "batch_size": _cfg_value(config, "batch_size"),
        "lr": _cfg_value(config, "lr"),
        "weight_decay": _cfg_value(config, "weight_decay"),
        "epoch_end_val_r2": summary.get("epoch_end_val_r2"),
        "val_r2_score": summary.get("val_r2_score"),
        "step_valid_loss": summary.get("step_valid_loss"),
        "valid_loss": summary.get("valid_loss"),
        "runtime_s": summary.get("_runtime"),
        "sweep_ids": _extract_sweep_ids(debug_log),
        "failure_type": _extract_failure_type(output_log),
        "has_traceback": "Traceback" in output_log,
        "best_ckpt_path": best_ckpt_path,
        "best_ckpt_exists": best_ckpt_exists,
        "config_path": str(run_dir / "files" / "config.yaml"),
    }


def _extract_prefixed_args(config: dict, prefix: str) -> dict:
    out = {}
    for key, raw in config.items():
        if not key.startswith(prefix):
            continue
        name = key[len(prefix):]
        if "/" in name:
            continue
        value = raw.get("value") if isinstance(raw, dict) and "value" in raw else raw
        out[name] = value
    return out


def _filter_ctor_kwargs(cls, kwargs: dict) -> dict:
    sig = inspect.signature(cls.__init__)
    allowed = {
        param.name
        for param in sig.parameters.values()
        if param.kind in (param.POSITIONAL_OR_KEYWORD, param.KEYWORD_ONLY)
    }
    allowed.discard("self")
    return {k: v for k, v in kwargs.items() if k in allowed}


def _unpack_batch_xy(batch):
    if isinstance(batch, (tuple, list)):
        if len(batch) == 2:
            return batch[0], batch[1]
        if len(batch) == 3:
            return batch[0], batch[1]
    raise ValueError(f"Unexpected batch structure: {type(batch)} / len={len(batch)}")


def _make_eval_dataloader(data_module, split_name: str):
    if split_name == "train":
        dataset = data_module._df_to_dataset(data_module.df_train, training=False)
    elif split_name == "val":
        dataset = data_module.dataset_val
    elif split_name == "test":
        dataset = data_module.dataset_test
    else:
        raise ValueError(f"Unknown split_name: {split_name}")

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=data_module.batch_size,
        shuffle=False,
        num_workers=data_module.num_workers,
        pin_memory=True,
    )


@torch.no_grad()
def _predict_split(dataloader, model, device):
    preds_list = []
    labels_list = []
    for batch in dataloader:
        x, y = _unpack_batch_xy(batch)
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        if pred.dim() == 2 and pred.shape[1] == 1:
            pred = pred.squeeze(1)
        if y.dim() == 2 and y.shape[1] == 1:
            y = y.squeeze(1)
        preds_list.append(pred.detach().cpu())
        labels_list.append(y.detach().cpu())
    preds = torch.cat(preds_list, dim=0).numpy().reshape(-1)
    labels = torch.cat(labels_list, dim=0).numpy().reshape(-1)
    return labels, preds


def _compute_regression_metrics(dataloader, model, device) -> dict:
    y_true, y_pred = _predict_split(dataloader, model, device)
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    ss_res = float(((y_true - y_pred) ** 2).sum())
    classical_r2 = np.nan if ss_tot < 1e-8 else float(1.0 - ss_res / ss_tot)

    if np.std(y_true) < 1e-8 or np.std(y_pred) < 1e-8:
        pearson_sq = np.nan
    else:
        pearson_sq = float(np.corrcoef(y_true, y_pred)[0, 1] ** 2)

    return {
        "r2_classic": classical_r2,
        "pearson_sq": pearson_sq,
    }

In [4]:
all_run_dirs = sorted([p for p in WANDB_ROOT.glob("run-*") if p.is_dir()])
records = [_run_record(p) for p in all_run_dirs]
runs_df = pd.DataFrame(records)

# Fallback mapping: sweep-*/config-<run_id>.yaml is often the most reliable local membership source.
run_to_sweeps = _build_run_to_sweeps_from_sweep_dir(WANDB_ROOT)
runs_df["sweep_ids_from_dirs"] = runs_df["run_id"].map(lambda rid: run_to_sweeps.get(rid, []))
runs_df["sweep_ids_merged"] = runs_df.apply(
    lambda r: sorted(set((r.get("sweep_ids") or []) + (r.get("sweep_ids_from_dirs") or []))),
    axis=1,
)

mode_frames = []
for mode, cfg in SWEEP_CONFIGS.items():
    sid = cfg["sweep_id"]
    lproj = cfg["logger_project"]
    metric = cfg["metric"]

    # Preferred filter: sweep id + logger_project
    df_mode = runs_df[
        runs_df["sweep_ids_merged"].map(lambda xs: sid in xs) & (runs_df["logger_project"] == lproj)
    ].copy()

    # Fallback if logger_project changed or was inconsistent
    if df_mode.empty:
        df_mode = runs_df[runs_df["sweep_ids_merged"].map(lambda xs: sid in xs)].copy()
        if not df_mode.empty:
            print(f"[warn] mode={mode}: no rows matched logger_project={lproj}; using sweep-id-only filter")

    df_mode["mode"] = mode
    df_mode["sweep_id"] = sid
    df_mode["metric_name"] = metric
    df_mode["logged_metric"] = df_mode[metric]
    mode_frames.append(df_mode)

compare_df = pd.concat(mode_frames, axis=0, ignore_index=True) if mode_frames else pd.DataFrame()

print(f"Total cached runs: {len(runs_df)}")
print("Runs selected per sweep:")
if compare_df.empty:
    print("No runs selected. Check SWEEP_CONFIGS values and local wandb cache.")
else:
    print(compare_df.groupby("mode")["run_id"].count())
    print("\nFailure type breakdown:")
    display(compare_df.groupby(["mode", "failure_type"]).size().rename("n_runs").reset_index())
    print("\nCheckpoint availability:")
    display(compare_df.groupby("mode")["best_ckpt_exists"].agg(["sum", "count"]))

    display_cols = [
        "mode",
        "run_id",
        "wandb_run_id",
        "logger_project",
        "model_module",
        "graph_module",
        "train_size_frac",
        "use_reverse_complements",
        "batch_size",
        "metric_name",
        "logged_metric",
        "failure_type",
        "best_ckpt_exists",
    ]
    display(compare_df[display_cols].sort_values(["mode", "logged_metric"], ascending=[True, False]).head(40))

Total cached runs: 144
Runs selected per sweep:
mode
basic       32
weighted    48
Name: run_id, dtype: int64

Failure type breakdown:


,mode,failure_type,n_runs
0,basic,none,32
1,weighted,none,44
2,weighted,tar_unexpected_end_of_data,4



Checkpoint availability:


,sum,count
mode,,
basic,0,32
weighted,0,48


,mode,run_id,wandb_run_id,logger_project,model_module,graph_module,train_size_frac,use_reverse_complements,batch_size,metric_name,logged_metric,failure_type,best_ckpt_exists
8,basic,upcxukbd,5723mawx,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.070860,none,False
17,basic,7jxn8jfy,54unr4fe,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,False,NaN,epoch_end_val_r2,0.057383,none,False
14,basic,tbhe7wjl,n7txyn4v,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.050187,none,False
16,basic,9flwuonw,cvaolwin,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.048764,none,False
5,basic,2q0j5ewl,ij2hgyvq,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.040457,none,False
10,basic,x88g7lgc,zvdgiq7q,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,False,NaN,epoch_end_val_r2,0.031020,none,False
7,basic,vfutsp60,r8gifdj3,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.030776,none,False
9,basic,wtqztpbw,g0kw8w0w,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.030041,none,False
4,basic,2q3gfllr,7riimj1y,bashor_lib1_scratch_basic,BassetVL,CNNBasicTraining,NaN,True,NaN,epoch_end_val_r2,0.029662,none,False
19,basic,w22wxnpo,xpzbjxrj,bashor_lib1_scratch_basic,ResNet1DRegressor,CNNBasicTraining,NaN,False,NaN,epoch_end_val_r2,0.028403,none,False


In [5]:
# Recover per-epoch metrics directly from W&B history.
# This works even when local checkpoint files have been deleted.

WANDB_ENTITY = os.environ.get("WANDB_ENTITY")  # optional manual override, e.g. "minhang"
WANDB_HISTORY_KEYS = [
    "_step",
    "epoch",
    "current_epoch",
    "trainer/global_step",
    "learning_rate",
    "train_loss",
    "valid_loss",
    "step_valid_loss",
    "valid_r2",
    "step_valid_r2",
    "valid_mean_pearson",
    "epoch_end_val_r2",
    "val_r2_score",
    "arithmetic_mean_loss",
    "harmonic_mean_loss",
    "prediction_mean_spearman",
    "entropy_spearman",
]
WANDB_HISTORY_MAX_RUNS_PER_MODE = None  # set to a small int for a quick smoke test

try:
    import wandb
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "wandb is not installed in this notebook kernel. Run `%pip install wandb` in this kernel, then re-run this cell."
    ) from exc

api = wandb.Api(timeout=60)


def _candidate_wandb_entities(compare_df: pd.DataFrame) -> list:
    candidates = []

    def _add(value):
        if not isinstance(value, str):
            return
        value = value.strip()
        if not value or value in candidates:
            return
        candidates.append(value)

    _add(WANDB_ENTITY)
    _add(os.environ.get("USER"))

    if "run_dir" in compare_df.columns:
        for run_dir in compare_df["run_dir"].dropna().unique().tolist():
            metadata = _read_json(Path(run_dir) / "files" / "wandb-metadata.json")
            _add(metadata.get("username"))
            email = metadata.get("email")
            if isinstance(email, str) and "@" in email:
                _add(email.split("@", 1)[0])

    return candidates


WANDB_ENTITY_CANDIDATES = _candidate_wandb_entities(compare_df)
print("Candidate W&B entities:", WANDB_ENTITY_CANDIDATES)

if not WANDB_ENTITY_CANDIDATES:
    raise ValueError(
        "Could not infer a W&B entity from the local cache. Set WANDB_ENTITY manually, for example `WANDB_ENTITY = \"minhang\"`."
    )


def _last_non_null(series: pd.Series):
    series = series.dropna()
    return series.iloc[-1] if len(series) else pd.NA


def _fetch_wandb_history(run_path: str, keys: list) -> pd.DataFrame:
    run = api.run(run_path)
    rows = list(run.scan_history(keys=keys, page_size=1000))
    hist = pd.DataFrame(rows)
    if hist.empty:
        return hist
    if "epoch" not in hist.columns and "current_epoch" in hist.columns:
        hist["epoch"] = hist["current_epoch"]
    hist["epoch"] = pd.to_numeric(hist.get("epoch"), errors="coerce")
    hist["_step"] = pd.to_numeric(hist.get("_step"), errors="coerce")
    hist["run_path"] = run_path
    hist["run_id"] = run.id
    hist["run_name"] = run.name
    return hist


history_status_rows = []
history_frames = []

for mode, df_mode in compare_df.groupby("mode", sort=True):
    if WANDB_HISTORY_MAX_RUNS_PER_MODE is not None:
        df_mode = df_mode.head(WANDB_HISTORY_MAX_RUNS_PER_MODE)

    for _, row in df_mode.iterrows():
        hist = None
        entity_used = None
        last_run_path = None
        last_exc = None
        remote_run_id = row.get("wandb_run_id") or row["run_id"]

        for entity in WANDB_ENTITY_CANDIDATES:
            run_path = f"{entity}/{row['logger_project']}/{remote_run_id}"
            last_run_path = run_path
            try:
                hist = _fetch_wandb_history(run_path, WANDB_HISTORY_KEYS)
                entity_used = entity
                break
            except Exception as exc:
                last_exc = exc

        if hist is None:
            history_status_rows.append(
                {
                    "mode": row["mode"],
                    "run_id": row["run_id"],
                    "wandb_run_id": remote_run_id,
                    "entity_used": None,
                    "run_path": last_run_path,
                    "n_history_rows": 0,
                    "history_error": str(last_exc) if last_exc is not None else "history_fetch_failed",
                }
            )
            continue

        if hist.empty:
            history_status_rows.append(
                {
                    "mode": row["mode"],
                    "run_id": row["run_id"],
                    "wandb_run_id": remote_run_id,
                    "entity_used": entity_used,
                    "run_path": last_run_path,
                    "n_history_rows": 0,
                    "history_error": "empty_history",
                }
            )
            continue

        hist["mode"] = row["mode"]
        hist["local_run_id"] = row["run_id"]
        hist["wandb_run_id"] = remote_run_id
        hist["logger_project"] = row["logger_project"]
        hist["model_module"] = row["model_module"]
        hist["graph_module"] = row["graph_module"]
        hist["metric_name"] = row["metric_name"]
        hist["entity_used"] = entity_used
        history_frames.append(hist)
        history_status_rows.append(
            {
                "mode": row["mode"],
                "run_id": row["run_id"],
                "wandb_run_id": remote_run_id,
                "entity_used": entity_used,
                "run_path": last_run_path,
                "n_history_rows": len(hist),
                "history_error": None,
            }
        )

wandb_history_status_df = pd.DataFrame(history_status_rows)
wandb_history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()

print("W&B history fetch status:")
display(wandb_history_status_df.head(20))
print(
    f"Runs with recovered history: {(wandb_history_status_df['history_error'].isna()).sum()} / {len(wandb_history_status_df)}"
)

if len(wandb_history_df) > 0:
    agg_map = {
        "_step": "max",
        "trainer/global_step": "max",
        "learning_rate": _last_non_null,
        "train_loss": _last_non_null,
        "valid_loss": _last_non_null,
        "step_valid_loss": _last_non_null,
        "valid_r2": _last_non_null,
        "step_valid_r2": _last_non_null,
        "valid_mean_pearson": _last_non_null,
        "epoch_end_val_r2": _last_non_null,
        "val_r2_score": _last_non_null,
        "arithmetic_mean_loss": _last_non_null,
        "harmonic_mean_loss": _last_non_null,
        "prediction_mean_spearman": _last_non_null,
        "entropy_spearman": _last_non_null,
    }
    present_agg_map = {k: v for k, v in agg_map.items() if k in wandb_history_df.columns}
    wandb_epoch_history_df = (
        wandb_history_df.sort_values(["mode", "run_id", "_step"])
        .groupby(["mode", "run_id", "run_name", "logger_project", "model_module", "graph_module", "metric_name", "epoch"], dropna=False, as_index=False)
        .agg(present_agg_map)
        .sort_values(["mode", "run_id", "epoch"])
    )

    print("Per-epoch history preview:")
    preview_cols = [
        c for c in [
            "mode",
            "run_id",
            "epoch",
            "train_loss",
            "valid_loss",
            "valid_r2",
            "epoch_end_val_r2",
            "val_r2_score",
        ] if c in wandb_epoch_history_df.columns
    ]
    display(wandb_epoch_history_df[preview_cols].head(20))

    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
        for ax, mode in zip(axes, ["basic", "weighted"]):
            metric_name = SWEEP_CONFIGS[mode]["metric"]
            mode_hist = wandb_epoch_history_df[wandb_epoch_history_df["mode"] == mode].copy()
            if mode_hist.empty or metric_name not in mode_hist.columns:
                ax.set_title(f"{mode}: no W&B history for {metric_name}")
                ax.set_xlabel("epoch")
                continue

            ranked_run_ids = (
                mode_hist.groupby("run_id")[metric_name]
                .max()
                .sort_values(ascending=False)
                .head(5)
                .index
            )
            for run_id in ranked_run_ids:
                run_hist = mode_hist[mode_hist["run_id"] == run_id].sort_values("epoch")
                run_hist = run_hist.dropna(subset=["epoch", metric_name])
                if len(run_hist) == 0:
                    continue
                ax.plot(run_hist["epoch"], run_hist[metric_name], marker="o", alpha=0.8, label=run_id)

            ax.set_title(f"{mode}: top-5 runs by {metric_name}")
            ax.set_xlabel("epoch")
            ax.set_ylabel(metric_name)
            ax.legend(title="run_id", fontsize=8)
            ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"W&B history plotting skipped: {exc}")
else:
    wandb_epoch_history_df = pd.DataFrame()
    print("No W&B history rows were recovered. Check WANDB_ENTITY and W&B authentication.")
    print("This path will recover per-epoch logged metrics, but not offline train/val/test re-evaluation if those were never logged.")

Candidate W&B entities: ['minhang', 'minhangjason1998']
W&B history fetch status:


,mode,run_id,wandb_run_id,entity_used,run_path,n_history_rows,history_error
0,basic,nmlx2b28,6naer7dd,None,minhangjason1998/bashor_lib1_scratch_basic/6na...,0,Could not find run <Run minhangjason1998/basho...
1,basic,fqtgotlu,404yplam,None,minhangjason1998/bashor_lib1_scratch_basic/404...,0,Could not find run <Run minhangjason1998/basho...
2,basic,z5fjo5ld,t248bqvl,None,minhangjason1998/bashor_lib1_scratch_basic/t24...,0,Could not find run <Run minhangjason1998/basho...
3,basic,osvntogl,bi9ysu68,None,minhangjason1998/bashor_lib1_scratch_basic/bi9...,0,Could not find run <Run minhangjason1998/basho...
4,basic,2q3gfllr,7riimj1y,None,minhangjason1998/bashor_lib1_scratch_basic/7ri...,0,Could not find run <Run minhangjason1998/basho...
5,basic,2q0j5ewl,ij2hgyvq,None,minhangjason1998/bashor_lib1_scratch_basic/ij2...,0,Could not find run <Run minhangjason1998/basho...
6,basic,sc58rd8p,abbqizh3,None,minhangjason1998/bashor_lib1_scratch_basic/abb...,0,Could not find run <Run minhangjason1998/basho...
7,basic,vfutsp60,r8gifdj3,None,minhangjason1998/bashor_lib1_scratch_basic/r8g...,0,Could not find run <Run minhangjason1998/basho...
8,basic,upcxukbd,5723mawx,None,minhangjason1998/bashor_lib1_scratch_basic/572...,0,Could not find run <Run minhangjason1998/basho...
9,basic,wtqztpbw,g0kw8w0w,None,minhangjason1998/bashor_lib1_scratch_basic/g0k...,0,Could not find run <Run minhangjason1998/basho...


Runs with recovered history: 0 / 80
No W&B history rows were recovered. Check WANDB_ENTITY and W&B authentication.
This path will recover per-epoch logged metrics, but not offline train/val/test re-evaluation if those were never logged.


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_rows = []
split_rows = []

for _, row in compare_df.iterrows():
    run_id = row["run_id"]
    mode = row["mode"]
    config_path = Path(row["config_path"])
    ckpt_path = row["best_ckpt_path"]

    if not isinstance(ckpt_path, str) or not Path(ckpt_path).is_file():
        eval_rows.append({"mode": mode, "run_id": run_id, "eval_error": "missing_checkpoint"})
        continue

    try:
        config = _read_yaml(config_path)
        data_module_name = _cfg_value(config, "data_module")
        model_module_name = _cfg_value(config, "model_module")

        Data = getattr(boda.data, data_module_name)
        Model = getattr(boda.model, model_module_name)

        data_kwargs = _filter_ctor_kwargs(Data, _extract_prefixed_args(config, "Data Module args."))
        model_kwargs = _filter_ctor_kwargs(Model, _extract_prefixed_args(config, "Model Module args."))

        data = Data(**data_kwargs)
        data.setup(stage="fit")
        data.setup(stage="test")

        train_eval_loader = _make_eval_dataloader(data, "train")
        val_eval_loader = _make_eval_dataloader(data, "val")
        test_eval_loader = _make_eval_dataloader(data, "test")

        split_rows.append({
            "mode": mode,
            "run_id": run_id,
            "n_train": int(len(data.df_train)),
            "n_val": int(len(data.df_val)),
            "n_test": int(len(data.df_test)),
            "batch_size": int(data.batch_size),
            "train_eval_batches": int(len(train_eval_loader)),
            "val_eval_batches": int(len(val_eval_loader)),
            "test_eval_batches": int(len(test_eval_loader)),
            "train_size_frac": data_kwargs.get("train_size_frac"),
            "test_min_barcodes": data_kwargs.get("test_min_barcodes"),
        })

        model = Model(**model_kwargs)
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["state_dict"]
        state_dict = {
            key[len("model."):] if key.startswith("model.") else key: value
            for key, value in state_dict.items()
            if key.startswith("model.")
        }
        model.load_state_dict(state_dict, strict=True)
        model = model.to(device)
        model.eval()

        train_metrics_eval = _compute_regression_metrics(train_eval_loader, model, device)
        val_metrics_eval = _compute_regression_metrics(val_eval_loader, model, device)
        test_metrics_eval = _compute_regression_metrics(test_eval_loader, model, device)

        eval_rows.append(
            {
                "mode": mode,
                "run_id": run_id,
                "model_module": row["model_module"],
                "graph_module": row["graph_module"],
                "train_size_frac": row["train_size_frac"],
                "train_min_barcodes": row["train_min_barcodes"],
                "use_reverse_complements": row["use_reverse_complements"],
                "batch_size": row["batch_size"],
                "lr": row["lr"],
                "weight_decay": row["weight_decay"],
                "metric_name": row["metric_name"],
                "logged_metric": row["logged_metric"],
                "eval_train_r2_classic": train_metrics_eval["r2_classic"],
                "eval_train_pearson_sq": train_metrics_eval["pearson_sq"],
                "eval_val_r2_classic": val_metrics_eval["r2_classic"],
                "eval_val_pearson_sq": val_metrics_eval["pearson_sq"],
                "eval_test_r2_classic": test_metrics_eval["r2_classic"],
                "eval_test_pearson_sq": test_metrics_eval["pearson_sq"],
                "failure_type": row["failure_type"],
                "best_ckpt_path": ckpt_path,
            }
        )
    except Exception as exc:
        eval_rows.append({"mode": mode, "run_id": run_id, "eval_error": str(exc)})

eval_df = pd.DataFrame(eval_rows)
split_df = pd.DataFrame(split_rows)

expected_eval_columns = [
    "mode",
    "run_id",
    "model_module",
    "graph_module",
    "train_size_frac",
    "train_min_barcodes",
    "use_reverse_complements",
    "batch_size",
    "lr",
    "weight_decay",
    "metric_name",
    "logged_metric",
    "eval_train_r2_classic",
    "eval_train_pearson_sq",
    "eval_val_r2_classic",
    "eval_val_pearson_sq",
    "eval_test_r2_classic",
    "eval_test_pearson_sq",
    "failure_type",
    "best_ckpt_path",
    "eval_error",
]
eval_df = eval_df.reindex(columns=expected_eval_columns)

n_ok = int(eval_df["eval_error"].isna().sum()) if "eval_error" in eval_df.columns else len(eval_df)
print(f"Evaluated runs successfully: {n_ok} / {len(eval_df)}")
print("\nSample split sanity (first 5 rows):")
display(split_df.head())
eval_df.head()

Evaluated runs successfully: 0 / 80

Sample split sanity (first 5 rows):


""


,mode,run_id,model_module,graph_module,train_size_frac,train_min_barcodes,use_reverse_complements,batch_size,lr,weight_decay,...,logged_metric,eval_train_r2_classic,eval_train_pearson_sq,eval_val_r2_classic,eval_val_pearson_sq,eval_test_r2_classic,eval_test_pearson_sq,failure_type,best_ckpt_path,eval_error
0,basic,nmlx2b28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing_checkpoint
1,basic,fqtgotlu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing_checkpoint
2,basic,z5fjo5ld,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing_checkpoint
3,basic,osvntogl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing_checkpoint
4,basic,2q3gfllr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing_checkpoint


In [7]:
print("Why val/test dataloader length can be 5:")
print("- len(dataloader) is number of batches, not rows")
print("- with ~307 rows and batch_size=64 => ceil(307/64)=5 batches")

if len(split_df) > 0:
    split_stats = (
        split_df.groupby("mode")[["n_train", "n_val", "n_test", "val_eval_batches", "test_eval_batches"]]
        .agg(["mean", "min", "max"])
    )
    display(split_stats)
else:
    print("split_df is empty (no evaluated runs)")

Why val/test dataloader length can be 5:
- len(dataloader) is number of batches, not rows
- with ~307 rows and batch_size=64 => ceil(307/64)=5 batches
split_df is empty (no evaluated runs)


In [8]:
if "eval_error" in eval_df.columns:
    error_df = eval_df[eval_df["eval_error"].notna()].copy()
    if len(error_df) > 0:
        print("Evaluation-time errors:")
        display(error_df[["mode", "run_id", "eval_error"]])

ok_df = eval_df[eval_df.get("eval_error", pd.Series([None] * len(eval_df))).isna()].copy()

if len(ok_df) == 0:
    print("No runs produced evaluable split metrics in this workspace.")
    print("All selected runs failed before re-loading a best checkpoint for train/val/test evaluation.")
    if "eval_error" in eval_df.columns:
        print("Evaluation error breakdown:")
        display(eval_df.groupby("eval_error").size().rename("n_runs").reset_index())
    print("Most likely cause here: local checkpoint files are no longer present, even though W&B summaries/logs are cached.")

print("Top runs by mode (evaluated test metrics):")
display(
    ok_df.sort_values(["mode", "eval_test_r2_classic"], ascending=[True, False])[
        [
            "mode",
            "run_id",
            "model_module",
            "graph_module",
            "train_size_frac",
            "use_reverse_complements",
            "batch_size",
            "metric_name",
            "logged_metric",
            "eval_train_r2_classic",
            "eval_train_pearson_sq",
            "eval_val_r2_classic",
            "eval_val_pearson_sq",
            "eval_test_r2_classic",
            "eval_test_pearson_sq",
        ]
    ].head(30)
)

print("Summary stats by mode:")
summary_by_mode = ok_df.groupby("mode")[[
    "eval_train_r2_classic",
    "eval_train_pearson_sq",
    "eval_val_r2_classic",
    "eval_val_pearson_sq",
    "eval_test_r2_classic",
    "eval_test_pearson_sq",
]].agg(["count", "mean", "median", "std", "max"])
display(summary_by_mode)

print("Summary stats by mode x model:")
summary_by_mode_model = ok_df.groupby(["mode", "model_module"])[[
    "eval_test_r2_classic",
    "eval_test_pearson_sq",
]].agg(["count", "mean", "median", "std", "max"])
display(summary_by_mode_model)

print("Failure breakdown from training logs (all selected runs):")
failure_breakdown = compare_df.groupby(["mode", "failure_type"]).size().rename("n_runs").reset_index()
display(failure_breakdown)

print("Note on 'OSError: unexpected end of data':")
print("- this is raised by Python tarfile while packing artifacts")
print("- training/checkpointing may finish, but tar packaging can fail near run end")
print("- those runs can still be evaluable if best checkpoint exists")

out_csv = LEARN_DIR / "run_registry" / f"lib1_compare_sweeps_{SWEEP_CONFIGS['basic']['sweep_id']}_vs_{SWEEP_CONFIGS['weighted']['sweep_id']}.csv"
ok_df.to_csv(out_csv, index=False)
print(f"Saved evaluated comparison results to: {out_csv}")

if len(ok_df) == 0:
    print("Skipping local test-metric boxplots because no runs had recoverable local checkpoints.")
else:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ok_df.boxplot(column="eval_test_r2_classic", by="mode", ax=ax[0])
        ax[0].set_title("Test classical R2 by mode")
        ax[0].set_xlabel("mode")
        ax[0].set_ylabel("eval_test_r2_classic")

        ok_df.boxplot(column="eval_test_pearson_sq", by="mode", ax=ax[1])
        ax[1].set_title("Test Pearson^2 by mode")
        ax[1].set_xlabel("mode")
        ax[1].set_ylabel("eval_test_pearson_sq")

        plt.suptitle("")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Plotting skipped: {exc}")

Evaluation-time errors:


,mode,run_id,eval_error
0,basic,nmlx2b28,missing_checkpoint
1,basic,fqtgotlu,missing_checkpoint
2,basic,z5fjo5ld,missing_checkpoint
3,basic,osvntogl,missing_checkpoint
4,basic,2q3gfllr,missing_checkpoint
...,...,...,...
75,weighted,1oawgy5k,missing_checkpoint
76,weighted,su93t2d1,missing_checkpoint
77,weighted,llr8khp0,missing_checkpoint
78,weighted,yq8stv19,missing_checkpoint


No runs produced evaluable split metrics in this workspace.
All selected runs failed before re-loading a best checkpoint for train/val/test evaluation.
Evaluation error breakdown:


,eval_error,n_runs
0,missing_checkpoint,80


Most likely cause here: local checkpoint files are no longer present, even though W&B summaries/logs are cached.
Top runs by mode (evaluated test metrics):


...

Summary stats by mode x model:


Empty DataFrame
Columns: [(eval_test_r2_classic, count), (eval_test_r2_classic, mean), (eval_test_r2_classic, median), (eval_test_r2_classic, std), (eval_test_r2_classic, max), (eval_test_pearson_sq, count), (eval_test_pearson_sq, mean), (eval_test_pearson_sq, median), (eval_test_pearson_sq, std), (eval_test_pearson_sq, max)]
Index: []

Failure breakdown from training logs (all selected runs):


,mode,failure_type,n_runs
0,basic,none,32
1,weighted,none,44
2,weighted,tar_unexpected_end_of_data,4


Note on 'OSError: unexpected end of data':
- this is raised by Python tarfile while packing artifacts
- training/checkpointing may finish, but tar packaging can fail near run end
- those runs can still be evaluable if best checkpoint exists
Saved evaluated comparison results to: /home/minhang/synBio_AL/boda2_EU/src/learn/run_registry/lib1_compare_sweeps_59fagojd_vs_tsdqtfnj.csv
Skipping local test-metric boxplots because no runs had recoverable local checkpoints.


In [9]:
if len(ok_df) == 0:
    print("Skipping local val-metric boxplots because no runs had recoverable local checkpoints.")
else:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ok_df.boxplot(column="eval_val_r2_classic", by="mode", ax=ax[0])
        ax[0].set_title("Val classical R2 by mode")
        ax[0].set_xlabel("mode")
        ax[0].set_ylabel("eval_val_r2_classic")

        ok_df.boxplot(column="eval_val_pearson_sq", by="mode", ax=ax[1])
        ax[1].set_title("Val Pearson^2 by mode")
        ax[1].set_xlabel("mode")
        ax[1].set_ylabel("eval_val_pearson_sq")

        plt.suptitle("")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Plotting skipped: {exc}")

Skipping local val-metric boxplots because no runs had recoverable local checkpoints.


In [10]:
if len(ok_df) == 0:
    print("Skipping architecture boxplots because no runs had recoverable local checkpoints.")
else:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ok_df.boxplot(column="eval_test_r2_classic", by="model_module", ax=ax[0])
        ax[0].set_title("Test classical R2 by architecture")
        ax[0].set_xlabel("model module")
        ax[0].set_ylabel("eval_test_r2_classic")

        ok_df.boxplot(column="eval_test_pearson_sq", by="model_module", ax=ax[1])
        ax[1].set_title("Test Pearson^2 by architecture")
        ax[1].set_xlabel("model module")
        ax[1].set_ylabel("eval_test_pearson_sq")

        plt.suptitle("")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Plotting skipped: {exc}")

Skipping architecture boxplots because no runs had recoverable local checkpoints.
